# Benchmarking Climate line plot

In [1]:
import sys, csv, json, gc
from pathlib import Path

sys.path.append("/home/jovyan")
from config_hf_token import HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)

ROOT_DIR = next(p for p in [Path().resolve(), *Path().resolve().parents]
                if (p / "benchmarking").is_dir())
sys.path.insert(0, str(ROOT_DIR / "benchmarking"))

from utils.quanti_benchmarking_1_details import ask_question, ask_question_mistral, ask_question_gemma
from utils.quanti_benchmarking_1_analysis import normalize_number

CLIMATE_DIR = ROOT_DIR / "climate_pilot"
OUTPUT_DIR = Path().resolve() / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

# ground truth, keyed by image number
ground_truth = {row["num"]: row for row in csv.DictReader(open(CLIMATE_DIR / "ground_truth_climate.csv"))}
print(f"{len(ground_truth)} ground-truth rows loaded")



25 ground-truth rows loaded


In [2]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, Qwen3VLForConditionalGeneration

MODEL_REGISTRY = [
    {"label": "Gemma-12B",       "slug": "gemma4-12b",      "checkpoint": "google/gemma-4-12B-it",                        "loader": "gemma"},
    {"label": "Gemma-E4B",       "slug": "gemma4-e4b",      "checkpoint": "google/gemma-4-E4B-it",                        "loader": "gemma"},
    {"label": "Ministral-3-14B", "slug": "ministral-3-14b", "checkpoint": "mistralai/Ministral-3-14B-Instruct-2512-BF16", "loader": "mistral"},
    {"label": "Ministral-3-8B",  "slug": "ministral-3-8b",  "checkpoint": "mistralai/Ministral-3-8B-Instruct-2512-BF16",  "loader": "mistral"},
    {"label": "Qwen3-VL-4B",     "slug": "qwen3-vl-4b",     "checkpoint": "Qwen/Qwen3-VL-4B-Instruct",                    "loader": "qwen"},
    {"label": "Qwen3-VL-8B",     "slug": "qwen3-vl-8b",     "checkpoint": "Qwen/Qwen3-VL-8B-Instruct",                    "loader": "qwen"},
]

ASK_FN = {"gemma": ask_question_gemma, "mistral": ask_question_mistral, "qwen": ask_question}


def load_model(entry):
    if entry["loader"] == "qwen":
        processor = AutoProcessor.from_pretrained(entry["checkpoint"])
        model = Qwen3VLForConditionalGeneration.from_pretrained(
            entry["checkpoint"], torch_dtype=torch.bfloat16, device_map="auto"
        ).eval()
    elif entry["loader"] == "mistral":
        processor = AutoProcessor.from_pretrained(entry["checkpoint"], fix_mistral_regex=True)
        model = AutoModelForImageTextToText.from_pretrained(
            entry["checkpoint"], torch_dtype=torch.bfloat16, device_map="auto"
        ).eval()
    else:  # gemma
        processor = AutoProcessor.from_pretrained(entry["checkpoint"])
        model = AutoModelForImageTextToText.from_pretrained(
            entry["checkpoint"], torch_dtype=torch.bfloat16, device_map="auto", attn_implementation="sdpa"
        ).eval()
    model.generation_config.max_length = None
    device = next(model.parameters()).device
    return model, processor, device


def unload_model(model, processor):
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()

In [3]:
QUESTIONS = {
    "post_claim_correct": "Does the text in the post accurately describe the chart? Reply with only 'correct' or 'incorrect'.",
    "solar_year10": "What is the Solar investment value at Year 10? Reply with just the number.",
    "wind_year10": "What is the Wind investment value at Year 10? Reply with just the number.",
}

images = []
for p in sorted((CLIMATE_DIR / "posts/correct/PNGs").glob("*.png")):
    images.append((p.stem, str(p), "correct"))
for p in sorted((CLIMATE_DIR / "posts/incorrect/PNGs").glob("*.png")):
    images.append((p.stem, str(p), "incorrect"))

print(f"{len(images)} images ({sum(1 for _,_,v in images if v=='correct')} correct, "
      f"{sum(1 for _,_,v in images if v=='incorrect')} incorrect)")

50 images (25 correct, 25 incorrect)


In [4]:
for entry in MODEL_REGISTRY:
    out_path = OUTPUT_DIR / f"climate_benchmark_{entry['slug']}.json"
    if out_path.exists():
        print(f"⏭ {entry['label']}: already done, skipping")
        continue

    print(f"\n{'='*60}\nLoading {entry['label']} ({entry['checkpoint']})\n{'='*60}")
    model, processor, device = load_model(entry)
    ask_fn = ASK_FN[entry["loader"]]

    results = []
    for image_name, image_path, variant in images:
        num = image_name.split("_")[0]
        answers = {}
        for q_key, q_text in QUESTIONS.items():
            answers[q_key] = ask_fn(image_path, q_text, model, processor, device)
        results.append({"image": image_name, "num": num, "variant": variant, "answers": answers})
        print(f"  {image_name}: {answers}")

    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"✅ Saved: {out_path}")

    unload_model(model, processor)

⏭ Gemma-12B: already done, skipping
⏭ Gemma-E4B: already done, skipping
⏭ Ministral-3-14B: already done, skipping
⏭ Ministral-3-8B: already done, skipping

Loading Qwen3-VL-4B (Qwen/Qwen3-VL-4B-Instruct)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

  001_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '54.6', 'wind_year10': '47.3'}
  002_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '40.9', 'wind_year10': '36.2'}
  003_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '30.4', 'wind_year10': '25.9'}
  004_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '39.7', 'wind_year10': '34.2'}
  005_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '31.2', 'wind_year10': '24.5'}
  006_remy_ashford_c: {'post_claim_correct': 'incorrect', 'solar_year10': '12.6', 'wind_year10': '7.1'}
  007_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '28.4', 'wind_year10': '20.9'}
  008_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '20.9', 'wind_year10': '14.8'}
  009_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '42.5', 'wind_year10': '36.8'}
  010_remy_ashford_c: {'post_claim_correct': 'incorrect', 'solar_year10'

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

  001_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '54.6', 'wind_year10': '47.3'}
  002_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '40.9', 'wind_year10': '36.2'}
  003_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '32.4', 'wind_year10': '25.9'}
  004_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '39.7', 'wind_year10': '34.2'}
  005_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '31.2', 'wind_year10': '24.5'}
  006_remy_ashford_c: {'post_claim_correct': 'incorrect', 'solar_year10': '12.6', 'wind_year10': '7.1'}
  007_remy_ashford_c: {'post_claim_correct': 'incorrect', 'solar_year10': '28.4', 'wind_year10': '20.9'}
  008_remy_ashford_c: {'post_claim_correct': 'incorrect', 'solar_year10': '20.9', 'wind_year10': '14.8'}
  009_remy_ashford_c: {'post_claim_correct': 'correct', 'solar_year10': '42.5', 'wind_year10': '36.8'}
  010_remy_ashford_c: {'post_claim_correct': 'incorrect', 'solar_yea

In [7]:
def score_model(slug):
    path = OUTPUT_DIR / f"climate_benchmark_{slug}.json"
    if not path.exists():
        return None
    results = json.loads(path.read_text())
    scores = {"post_claim_correct": [], "solar_year10": [], "wind_year10": []}
    for r in results:
        gt = ground_truth[r["num"]]
        pred_claim = r["answers"]["post_claim_correct"].strip().lower()
        scores["post_claim_correct"].append(pred_claim == r["variant"])

        pred_solar = normalize_number(r["answers"]["solar_year10"])
        true_solar = normalize_number(gt["solar_final_year"])
        scores["solar_year10"].append(pred_solar == true_solar)

        pred_wind = normalize_number(r["answers"]["wind_year10"])
        true_wind = normalize_number(gt["wind_final_year"])
        scores["wind_year10"].append(pred_wind == true_wind)
    return {q: 100 * sum(v) / len(v) for q, v in scores.items()}


print(f"{'model':17s}{'post_claim_correct':>20s}{'solar_year10':>16s}{'wind_year10':>14s}")
for entry in MODEL_REGISTRY:
    acc = score_model(entry["slug"])
    if acc is None:
        print(f"{entry['label']:17s}  no data")
        continue
    print(f"{entry['label']:17s}{acc['post_claim_correct']:20.2f}{acc['solar_year10']:16.2f}{acc['wind_year10']:14.2f}")

model              post_claim_correct    solar_year10   wind_year10
Gemma-12B                       94.00           76.00         88.00
Gemma-E4B                       50.00           78.00         74.00
Ministral-3-14B                 50.00           96.00        100.00
Ministral-3-8B                  72.00           96.00         96.00
Qwen3-VL-4B                     58.00           92.00        100.00
Qwen3-VL-8B                     80.00           96.00         96.00


In [9]:
# checking output formats
def verify_no_format_artifacts(slug):
    """Confirms wrong answers are genuine content errors, not scoring artifacts from verbose/
    malformed model output -- checks (a) every post_claim_correct answer is a bare 'correct'/
    'incorrect', never verbose text, and (b) every wrong numeric answer is a cleanly-parseable
    number that's just factually off, not something normalize_number failed to extract.
    """
    path = OUTPUT_DIR / f"climate_benchmark_{slug}.json"
    if not path.exists():
        print(f"  ⚠ no data for {slug}")
        return
    results = json.loads(path.read_text())

    claim_answers = [r["answers"]["post_claim_correct"] for r in results]
    non_bare = [a for a in claim_answers if a.strip().lower() not in ("correct", "incorrect")]
    print(f"{slug}: post_claim_correct — distinct raw strings: {set(claim_answers)}, "
          f"non-bare-word count: {len(non_bare)}")
    if non_bare:
        print(f"   ⚠ verbose/malformed answers found: {non_bare[:3]}")

    for q, gt_key in [("solar_year10", "solar_final_year"), ("wind_year10", "wind_final_year")]:
        wrong = [(r["num"], r["answers"][q], ground_truth[r["num"]][gt_key]) for r in results
                 if normalize_number(r["answers"][q]) != normalize_number(ground_truth[r["num"]][gt_key])]
        if wrong:
            print(f"   {q}: {len(wrong)} wrong, all cleanly-formatted numbers (not extraction failures):")
            for num, raw, true in wrong[:3]:
                print(f"      {num}: said {raw!r}, actual {true!r}")


for entry in MODEL_REGISTRY:
    verify_no_format_artifacts(entry["slug"])
    print()


gemma4-12b: post_claim_correct — distinct raw strings: {'correct', 'incorrect'}, non-bare-word count: 0
   solar_year10: 12 wrong, all cleanly-formatted numbers (not extraction failures):
      003: said '33.4', actual '32.4'
      015: said '49.1', actual '46.1'
      018: said '51.2', actual '51.0'
   wind_year10: 6 wrong, all cleanly-formatted numbers (not extraction failures):
      004: said '31.2', actual '34.2'
      013: said '75.9', actual '75.0'
      025: said '51.3', actual '54.3'

gemma4-e4b: post_claim_correct — distinct raw strings: {'correct'}, non-bare-word count: 0
   solar_year10: 11 wrong, all cleanly-formatted numbers (not extraction failures):
      001: said '51.6', actual '54.6'
      003: said '34.4', actual '32.4'
      015: said '40.4', actual '46.1'
   wind_year10: 13 wrong, all cleanly-formatted numbers (not extraction failures):
      006: said '12.6', actual '7.1'
      009: said '42.5', actual '36.8'
      016: said '20.3', actual '14.6'

ministral-3-14b

## Running v1-v6 post_claim prompt options to compare the performance of the models to the spotify plot benchmark results

In [10]:
CLIMATE_PROMPT_VERSIONS = {
    "v1": "Does the text in the post accurately describe the chart? Reply with only 'correct' or 'incorrect'.",
    "v2": "Read the text in the post and look at the chart. Does the text correctly describe what the chart shows? Reply with only 'correct' or 'incorrect'.",
    "v3": "The post text claims one energy source ended up with higher investment than the other by Year 10. Look at the values in the chart to verify this claim. If the energy source described as higher in the text has a higher value at Year 10 in the chart, reply 'correct'. If not, reply 'incorrect'.",
    "v4": "In the post, the text above the image makes a claim comparing Solar and Wind investment by Year 10. Identify the according values in the chart to verify this claim. If the claim matches the visualization reply 'correct'. If not, reply 'incorrect'.",
    "v5": "In the post, the text above the chart claims one energy source ended up higher than the other by Year 10. Find the Year 10 value for Solar and the Year 10 value for Wind in the chart. If the energy source the text says is higher has a numerically higher Year 10 value, reply 'correct'. If not, reply 'incorrect'.",
    "v6": "Look at the text above the chart. It makes a claim about Solar and Wind investment by Year 10. Step 1: find the Year 10 value for Solar in the chart. Step 2: find the Year 10 value for Wind in the chart. Step 3: check if the claim in the text matches which one is higher. If it matches, reply 'correct'. If not, reply 'incorrect'.",
}


for entry in MODEL_REGISTRY:
    out_path = OUTPUT_DIR / f"climate_promptversions_{entry['slug']}.json"
    if out_path.exists():
        print(f"⏭ {entry['label']}: already done, skipping")
        continue

    print(f"\n{'='*60}\nLoading {entry['label']} ({entry['checkpoint']})\n{'='*60}")
    model, processor, device = load_model(entry)
    ask_fn = ASK_FN[entry["loader"]]

    results = []
    for image_name, image_path, variant in images:
        num = image_name.split("_")[0]
        for v_key, v_text in CLIMATE_PROMPT_VERSIONS.items():
            answer = ask_fn(image_path, v_text, model, processor, device)
            results.append({"image": image_name, "num": num, "variant": variant,
                             "prompt_version": v_key, "answer": answer})
        print(f"  {image_name}: done")

    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"✅ Saved: {out_path}")

    unload_model(model, processor)


Loading Gemma-12B (google/gemma-4-12B-it)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

  001_remy_ashford_c: done
  002_remy_ashford_c: done
  003_remy_ashford_c: done


KeyboardInterrupt: 

In [ ]:
def score_prompt_versions(slug):
    path = OUTPUT_DIR / f"climate_promptversions_{slug}.json"
    if not path.exists():
        return None
    results = json.loads(path.read_text())
    by_version = {}
    for r in results:
        by_version.setdefault(r["prompt_version"], []).append(r["answer"].strip().lower() == r["variant"])
    return {v: 100 * sum(vals) / len(vals) for v, vals in by_version.items()}


print(f"{'model':17s}" + "".join(f"v{i:<8}" for i in range(1, 7)))
for entry in MODEL_REGISTRY:
    acc = score_prompt_versions(entry["slug"])
    if acc is None:
        print(f"{entry['label']:17s}  no data")
        continue
    row = "".join(f"{acc.get(f'v{i}', float('nan')):8.2f} " for i in range(1, 7))
    print(f"{entry['label']:17s}{row}")

## Comparing v1-v6 post_claim performance when the content of the post differs - original spotify pie chart vs climate line chart

In [ ]:
import csv

# climate slug -> original-study benchmarking/ folder name (they differ for one model)
ORIGINAL_SLUG = {
    "gemma4-12b": "gemma4-12b",
    "gemma4-e4b": "gemma-e4b",
    "ministral-3-14b": "ministral-3-14b",
    "ministral-3-8b": "ministral-3-8b",
    "qwen3-vl-4b": "qwen3-vl-4b",
    "qwen3-vl-8b": "qwen3-vl-8b",
}


def score_original_test2(model_dir_name):
    """Original study's Test 2 (claim-only) per-version post_claim_correct accuracy, n=100 --
    the fair comparison point for climate's n=50, not Test 1's n=4."""
    scores = {}
    for v in range(1, 7):
        fp = (ROOT_DIR / "benchmarking" / "outputs" / model_dir_name / "quantitative"
              / "test-2-gn-claim-only" / f"v{v}" / f"accuracy_scores_v{v}.csv")
        if not fp.exists():
            continue
        rows = list(csv.DictReader(open(fp)))
        if not rows:
            continue
        col = "post_claim_correct_correct" if "post_claim_correct_correct" in rows[0] else "correct"
        vals = [r[col] == "True" for r in rows]
        if vals:
            scores[f"v{v}"] = 100 * sum(vals) / len(vals)
    return scores


print(f"{'model':17s}{'tree':10s}" + "".join(f"v{i:<8}" for i in range(1, 7)))
for entry in MODEL_REGISTRY:
    climate_acc = score_prompt_versions(entry["slug"]) or {}
    original_acc = score_original_test2(ORIGINAL_SLUG[entry["slug"]])

    climate_row = "".join(f"{climate_acc.get(f'v{i}', float('nan')):8.2f} " for i in range(1, 7))
    original_row = "".join(f"{original_acc.get(f'v{i}', float('nan')):8.2f} " for i in range(1, 7))
    print(f"{entry['label']:17s}{'climate':10s}{climate_row}")
    print(f"{'':17s}{'original':10s}{original_row}")
    print()